1) 문서 로더의 구조 이해하기

In [ ]:
# [목적] LangChain Document 객체로 텍스트 문서 한 건을 만드는 예제
# Document는 본문과 부가 정보를 함께 담는 형식이며, 이후 문서 로더와 검색 기능의 기본 단위로 사용됩니다.
# 이 셀에서는 본문만 넣어 객체를 먼저 만들고, 다음 셀에서 내부 구조와 메타데이터를 살펴봅니다.
from langchain_core.documents import Document

document = Document(page_content="안녕하세요? 이건 객체의 컨텐츠입니다")

In [ ]:
# [목적] 생성한 Document 객체에 어떤 정보가 들어 있는지 확인하는 예제
# __dict__는 객체가 보관하는 속성을 딕셔너리 형태로 보여 줍니다.
# 본문(page_content)과 메타데이터(metadata)가 어떻게 구성되는지 확인하기 위해 사용합니다.
document.__dict__

In [ ]:
# [목적] Document에 출처·페이지·작성자 정보를 메타데이터로 추가하는 예제
# metadata는 본문 외의 설명 정보를 저장하는 딕셔너리로, 나중에 검색 결과의 출처를 표시할 때 활용할 수 있습니다.
# 키별로 값을 넣은 뒤 마지막 줄에서 저장된 정보를 확인합니다.
document.metadata["source"] = "TeddyNote"
document.metadata["page"] = 1
document.metadata["author"] = "Teddy"

document.metadata  # 문서의 부가 정보 확인

In [ ]:
# [목적] 불러올 PDF 파일의 위치를 한 곳에 지정하는 예제
# FILE_PATH 변수에 경로를 저장해 이후 로더 생성과 문서 분할 과정에서 같은 파일을 일관되게 사용합니다.
# 파일이 바뀌면 이 값만 수정하면 되므로 경로 관리가 간단해집니다.
FILE_PATH = "../data/SPRI_AI_Brief_2023년12월호_F.pdf"

In [ ]:
# [목적] PyPDFLoader로 PDF를 읽을 준비를 하는 예제
# PyPDFLoader는 PDF의 각 페이지를 LangChain Document 목록으로 바꾸는 전용 로더입니다.
# 여기서는 경로를 전달해 로더만 만들고, 실제 읽기는 다음 셀의 load()에서 수행합니다.
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(FILE_PATH)  # PDF 로더 설정

In [ ]:
# [목적] 준비한 로더로 PDF 전체를 읽고 문서 개수를 확인하는 예제
# load()는 PDF 페이지를 Document 객체 목록으로 변환하며, docs 변수에 그 결과를 저장합니다.
# 페이지 수를 확인해 파일이 기대한 만큼 읽혔는지 점검합니다.
docs = loader.load()  # PDF 문서 로드

len(docs)  # 로드한 문서의 개수 확인

In [ ]:
# [목적] PDF에서 읽어 온 특정 페이지의 Document 내용을 확인하는 예제
# docs는 페이지별 Document 목록이므로 대괄호 인덱스로 원하는 페이지를 선택할 수 있습니다.
# 추출된 본문과 페이지 메타데이터가 올바른지 간단히 검토하기 위해 사용합니다.
docs[1]  # 두 번째 문서 확인

In [ ]:
# [목적] PDF 내용을 일정한 길이의 작은 문서 조각으로 나누는 예제
# RecursiveCharacterTextSplitter는 문단·문장 경계를 가능한 한 유지하며 긴 텍스트를 chunk_size 기준으로 분할합니다.
# 이렇게 만든 조각은 임베딩 생성과 유사도 검색에서 필요한 크기의 검색 단위로 사용됩니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 문자 수를 기준으로 문서를 나누는 도구 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=0
)

# 예제 파일 경로
FILE_PATH = "../data/SPRI_AI_Brief_2023년12월호_F.pdf"

# PDF 로더 설정
loader = PyPDFLoader(FILE_PATH)

# 문서 분할
split_docs = loader.load_and_split(text_splitter=text_splitter)

In [ ]:
# [목적] 분할된 문서의 개수와 한 조각의 내용을 확인하는 예제
# len()으로 생성된 조각 수를 확인하고, split_docs[10]으로 중간 조각 하나를 살펴봅니다.
# 분할 길이와 내용이 의도에 맞는지 검토해 이후 검색 품질을 조정하는 기준으로 사용합니다.
print(f"문서의 길이: {len(split_docs)}")  # 로드한 문서의 개수 확인

split_docs[10]  # 청크 문서 확인

In [ ]:
# [목적] PDF를 한 번에 모두 읽지 않고 페이지 단위로 순차 처리하는 예제
# lazy_load()는 필요할 때마다 Document를 꺼내는 반복 가능한 흐름을 제공해 큰 파일의 메모리 사용을 줄입니다.
# 반복문에서 각 문서의 metadata를 출력해 페이지별 정보를 순서대로 확인합니다.
loader.lazy_load()

for doc in loader.lazy_load():
    print(doc.metadata)

In [ ]:
# [목적] 비동기 PDF 로딩 호출이 반환하는 작업 객체를 확인하는 예제
# aload()는 비동기로 문서를 읽는 메서드이며, await 없이 호출하면 아직 완료되지 않은 코루틴 객체가 반환됩니다.
# 다음 셀에서 await를 사용해 실제 문서 목록을 받는 방식과 비교합니다.
adocs = loader.aload()

In [ ]:
# [목적] 비동기 방식으로 PDF를 읽어 실제 Document 목록을 받는 예제
# await는 비동기 로딩이 끝날 때까지 기다린 뒤 결과를 adocs에 저장합니다.
# 다른 비동기 작업과 함께 문서를 처리해야 할 때 이 방식을 사용할 수 있습니다.
adocs = await loader.aload()